# 04b - Proper Superposition-Inspired Experience Replay

**Phase 1: CartPole Environment**

## Objective

This notebook implements a **properly quantum-inspired** experience replay buffer based on **quantum superposition and interference**.

## Quantum Principle

In quantum mechanics:
1. **Superposition**: A particle can exist in multiple states simultaneously
2. **Interference**: When states combine, aligned phases amplify, opposite phases cancel

## Classical Translation

We implement:
1. **Prioritized sampling** - Like quantum amplitude (higher TD-error = higher amplitude)
2. **Phase-based combination** - Experiences with similar outcomes reinforce each other
3. **Interference weighting** - Agreeing experiences amplify, disagreeing cancel

## Key Differences from Original Notebook (04)

| Aspect | Original (04) | This Notebook (04b) |
|--------|---------------|---------------------|
| Combination | Simple average | Interference-based weighting |
| Priorities | None or basic | TD-error based (like amplitude) |
| Phase | Not used | Value-based phase alignment |
| Expected | Weak improvement | Should help more |

## Configuration

Same as baseline:
- `stoch_dim=64, deter_dim=512, hidden_dim=512`
- `batch_size=32, seq_len=20, lr=3e-4`
- `EXPERIMENT_SEEDS=[42, 123, 456, 789, 1024]`

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import json
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional, NamedTuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Import quantum-inspired buffer
from quantum_inspired import SuperpositionReplayBuffer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# Standard configuration
EXPERIMENT_SEEDS = [42, 123, 456, 789, 1024]

OBS_DIM = 4
ACTION_DIM = 2
STOCH_DIM = 64
DETER_DIM = 512
HIDDEN_DIM = 512

NUM_EPISODES = 100
NUM_STEPS = 10000
BATCH_SIZE = 32
SEQ_LEN = 20
LEARNING_RATE = 3e-4

# Superposition buffer parameters
BUFFER_ALPHA = 0.6       # Priority exponent
NUM_SUPERPOSE = 3        # Number of experiences to combine
INTERFERENCE_STRENGTH = 0.5  # How much interference affects weights

print("Configuration:")
print(f"  Buffer: alpha={BUFFER_ALPHA}, num_superpose={NUM_SUPERPOSE}")
print(f"  Interference strength: {INTERFERENCE_STRENGTH}")

In [ ]:
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 2. Data Collection

In [ ]:
def collect_episodes(env_name: str, num_episodes: int, seed: int) -> List[Dict]:
    """Collect episodes in format expected by SuperpositionReplayBuffer."""
    env = gym.make(env_name)
    episodes = []
    
    for ep_idx in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep_idx)
        
        observations = [obs]
        actions = []
        rewards = []
        
        done = False
        while not done:
            action = env.action_space.sample()
            action_onehot = np.zeros(ACTION_DIM)
            action_onehot[action] = 1.0
            
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            observations.append(obs)
            actions.append(action_onehot)
            rewards.append(reward)
        
        observations = observations[:-1]
        
        if len(observations) > 0:
            episodes.append({
                'obs': np.array(observations, dtype=np.float32),
                'actions': np.array(actions, dtype=np.float32),
                'rewards': np.array(rewards, dtype=np.float32)
            })
    
    env.close()
    return episodes

## 3. World Model (Same as Baseline)

In [ ]:
class RSSMState(NamedTuple):
    """RSSM state representation."""
    deter: torch.Tensor  # (batch, deter_dim)
    stoch: torch.Tensor  # (batch, stoch_dim)
    
    @property
    def combined(self) -> torch.Tensor:
        return torch.cat([self.deter, self.stoch], dim=-1)


class WorldModel(nn.Module):
    """
    RSSM-based World Model (DreamerV3 style).
    
    Architecture matches baseline exactly:
    - encoder: [512, 512] hidden layers
    - decoder: [512, 512] hidden layers  
    - reward_predictor: [512, 512] hidden layers
    - continue_predictor: [512, 512] hidden layers
    """
    
    def __init__(
        self,
        obs_dim: int = OBS_DIM,
        action_dim: int = ACTION_DIM,
        stoch_dim: int = STOCH_DIM,
        deter_dim: int = DETER_DIM,
        hidden_dim: int = HIDDEN_DIM
    ):
        super().__init__()
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.stoch_dim = stoch_dim
        self.deter_dim = deter_dim
        self.hidden_dim = hidden_dim
        self.state_dim = stoch_dim + deter_dim
        
        # Encoder: obs -> embedding [512, 512]
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU()
        )
        
        # Recurrent core
        self.input_proj = nn.Sequential(
            nn.Linear(stoch_dim + action_dim, hidden_dim),
            nn.ELU()
        )
        self.gru = nn.GRUCell(hidden_dim, deter_dim)
        
        # Prior (imagination): deter -> stoch
        self.prior_net = nn.Sequential(
            nn.Linear(deter_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, stoch_dim * 2)
        )
        
        # Posterior (inference): deter + embed -> stoch
        self.posterior_net = nn.Sequential(
            nn.Linear(deter_dim + hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, stoch_dim * 2)
        )
        
        # Decoder: state -> obs [512, 512]
        self.decoder = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, obs_dim)
        )
        
        # Reward predictor [512, 512]
        self.reward_predictor = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Continue predictor [512, 512]
        self.continue_predictor = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def initial_state(self, batch_size: int) -> RSSMState:
        return RSSMState(
            deter=torch.zeros(batch_size, self.deter_dim, device=DEVICE),
            stoch=torch.zeros(batch_size, self.stoch_dim, device=DEVICE)
        )
    
    def _get_dist(self, stats: torch.Tensor) -> torch.distributions.Normal:
        mean, log_std = stats.chunk(2, dim=-1)
        std = F.softplus(log_std) + 0.1
        return torch.distributions.Normal(mean, std)
    
    def observe(self, obs: torch.Tensor, action: torch.Tensor, state: RSSMState):
        embed = self.encoder(obs)
        x = self.input_proj(torch.cat([state.stoch, action], dim=-1))
        deter = self.gru(x, state.deter)
        
        posterior_stats = self.posterior_net(torch.cat([deter, embed], dim=-1))
        posterior_dist = self._get_dist(posterior_stats)
        stoch = posterior_dist.rsample()
        
        prior_stats = self.prior_net(deter)
        prior_dist = self._get_dist(prior_stats)
        
        return RSSMState(deter=deter, stoch=stoch), prior_dist, posterior_dist
    
    def imagine(self, action: torch.Tensor, state: RSSMState):
        x = self.input_proj(torch.cat([state.stoch, action], dim=-1))
        deter = self.gru(x, state.deter)
        
        prior_stats = self.prior_net(deter)
        prior_dist = self._get_dist(prior_stats)
        stoch = prior_dist.rsample()
        
        return RSSMState(deter=deter, stoch=stoch), prior_dist
    
    def decode(self, state: RSSMState) -> torch.Tensor:
        return self.decoder(state.combined)
    
    def predict_reward(self, state: RSSMState) -> torch.Tensor:
        return self.reward_predictor(state.combined).squeeze(-1)
    
    def predict_continue(self, state: RSSMState) -> torch.Tensor:
        return torch.sigmoid(self.continue_predictor(state.combined).squeeze(-1))
    
    def forward(self, obs_seq: torch.Tensor, action_seq: torch.Tensor) -> Dict[str, torch.Tensor]:
        batch_size, seq_len = obs_seq.shape[:2]
        state = self.initial_state(batch_size)
        
        recon_obs, pred_rewards, pred_continues = [], [], []
        priors, posteriors = [], []
        
        for t in range(seq_len):
            state, prior, posterior = self.observe(obs_seq[:, t], action_seq[:, t], state)
            recon_obs.append(self.decode(state))
            pred_rewards.append(self.predict_reward(state))
            pred_continues.append(self.predict_continue(state))
            priors.append(prior)
            posteriors.append(posterior)
        
        return {
            'recon_obs': torch.stack(recon_obs, dim=1),
            'pred_rewards': torch.stack(pred_rewards, dim=1),
            'pred_continues': torch.stack(pred_continues, dim=1),
            'priors': priors,
            'posteriors': posteriors
        }

## 4. Training with Superposition Buffer

In [ ]:
def compute_loss(output, obs_seq, reward_seq, kl_weight=1.0):
    recon_loss = F.mse_loss(output['recon_obs'], obs_seq)
    reward_loss = F.mse_loss(output['pred_rewards'], reward_seq)
    
    kl_losses = []
    for prior, posterior in zip(output['priors'], output['posteriors']):
        kl = torch.distributions.kl_divergence(posterior, prior).sum(-1).mean()
        kl_losses.append(kl)
    kl_loss = torch.stack(kl_losses).mean()
    
    total = recon_loss + reward_loss + kl_weight * kl_loss
    return {'total': total, 'recon': recon_loss, 'reward': reward_loss, 'kl': kl_loss}

In [ ]:
def train_with_superposition(
    model: WorldModel,
    buffer: SuperpositionReplayBuffer,
    num_steps: int = NUM_STEPS,
    verbose: bool = True
) -> pd.DataFrame:
    """Train using superposition replay buffer with interference."""
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history = []
    
    for step in range(num_steps):
        model.train()
        
        # Sample with superposition (combines multiple experiences with interference)
        batch = buffer.sample(batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
        
        obs = batch['obs'].to(DEVICE)
        actions = batch['actions'].to(DEVICE)
        rewards = batch['rewards'].to(DEVICE)
        weights = batch['weights'].to(DEVICE)  # Importance sampling weights
        
        optimizer.zero_grad()
        output = model(obs, actions)
        losses = compute_loss(output, obs, rewards)
        
        # Weight loss by importance sampling weights
        weighted_loss = (losses['total'] * weights.mean())
        weighted_loss.backward()
        
        optimizer.step()
        
        # Update priorities based on reconstruction error (TD-error proxy)
        with torch.no_grad():
            td_errors = (output['recon_obs'] - obs).pow(2).mean(dim=(1, 2)).cpu().numpy()
        buffer.update_priorities(batch['indices'], td_errors.tolist())
        
        history.append({
            'step': step,
            'total': losses['total'].item(),
            'recon': losses['recon'].item(),
            'kl': losses['kl'].item(),
            'beta': buffer.beta
        })
        
        if verbose and step % 1000 == 0:
            print(f"Step {step}: total={losses['total'].item():.4f}, "
                  f"recon={losses['recon'].item():.4f}, beta={buffer.beta:.3f}")
    
    return pd.DataFrame(history)

## 5. Evaluation

In [ ]:
def evaluate_model(model, episodes, num_samples=100):
    """Evaluate on episodes."""
    model.eval()
    
    # Create simple buffer for evaluation
    eval_buffer = SuperpositionReplayBuffer(capacity=500, num_superpose=1)  # No superposition for eval
    for ep in episodes[:50]:
        eval_buffer.add(ep)
    
    batch = eval_buffer.sample_standard(batch_size=num_samples, seq_len=SEQ_LEN)
    obs = batch['obs'].to(DEVICE)
    actions = batch['actions'].to(DEVICE)
    
    with torch.no_grad():
        output = model(obs, actions)
        mse = F.mse_loss(output['recon_obs'], obs).item()
    
    return {'recon_mse': mse}


def evaluate_test_set(model, seed):
    test_episodes = collect_episodes('CartPole-v1', 50, seed=seed + 10000)
    return evaluate_model(model, test_episodes)

## 6. Run Experiments

In [ ]:
def run_single_experiment(seed: int, verbose: bool = True) -> Dict:
    print(f"\n{'='*60}")
    print(f"Running superposition experiment with seed {seed}")
    print(f"{'='*60}")
    
    set_seed(seed)
    start_time = time.time()
    
    # Collect data
    episodes = collect_episodes('CartPole-v1', NUM_EPISODES, seed=seed)
    
    # Create superposition buffer
    buffer = SuperpositionReplayBuffer(
        capacity=1000,
        alpha=BUFFER_ALPHA,
        num_superpose=NUM_SUPERPOSE,
        interference_strength=INTERFERENCE_STRENGTH
    )
    for ep in episodes:
        buffer.add(ep, td_error=1.0)  # Initial priority
    
    print(f"Buffer size: {len(buffer)} episodes")
    
    # Create and train model
    model = WorldModel().to(DEVICE)
    history = train_with_superposition(model, buffer, verbose=verbose)
    
    training_time = time.time() - start_time
    
    # Evaluate
    train_metrics = evaluate_model(model, episodes)
    test_metrics = evaluate_test_set(model, seed)
    
    print(f"\nResults: train_mse={train_metrics['recon_mse']:.6f}, test_mse={test_metrics['recon_mse']:.6f}")
    
    return {
        'seed': seed,
        'history': history,
        'final_loss': history['total'].iloc[-1],
        'train_mse': train_metrics['recon_mse'],
        'test_mse': test_metrics['recon_mse'],
        'training_time': training_time,
        'buffer_stats': buffer.get_stats()
    }

In [ ]:
print("="*60)
print("SUPERPOSITION REPLAY BUFFER EXPERIMENTS")
print("="*60)

results = []
for seed in EXPERIMENT_SEEDS:
    result = run_single_experiment(seed)
    results.append(result)

## 7. Results

In [ ]:
final_losses = [r['final_loss'] for r in results]
train_mses = [r['train_mse'] for r in results]
test_mses = [r['test_mse'] for r in results]

print("\n" + "="*60)
print("AGGREGATED RESULTS (Superposition Replay)")
print("="*60)
print(f"Final Loss: {np.mean(final_losses):.4f} +/- {np.std(final_losses):.4f}")
print(f"Train MSE:  {np.mean(train_mses):.6f} +/- {np.std(train_mses):.6f}")
print(f"Test MSE:   {np.mean(test_mses):.6f} +/- {np.std(test_mses):.6f}")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Training curves
ax = axes[0]
for r in results:
    ax.plot(r['history']['total'], alpha=0.5, label=f"Seed {r['seed']}")
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()

# Beta annealing
ax = axes[1]
for r in results:
    ax.plot(r['history']['beta'], alpha=0.5)
ax.set_xlabel('Step')
ax.set_ylabel('Beta')
ax.set_title('Importance Sampling Beta')

# Test MSE
ax = axes[2]
ax.bar(range(len(EXPERIMENT_SEEDS)), test_mses)
ax.axhline(np.mean(test_mses), color='r', linestyle='--')
ax.set_xticks(range(len(EXPERIMENT_SEEDS)))
ax.set_xticklabels(EXPERIMENT_SEEDS)
ax.set_xlabel('Seed')
ax.set_ylabel('Test MSE')
ax.set_title('Test MSE by Seed')

plt.tight_layout()
plt.savefig('../experiments/results/superposition_proper.png', dpi=150)
plt.show()

## 8. Save Results

In [ ]:
complete_metrics = {
    'approach': 'superposition_proper',
    'description': 'Superposition-inspired replay with interference-based combination',
    'quantum_principle': 'Superposition explores multiple states; interference amplifies agreement',
    'config': {
        'stoch_dim': STOCH_DIM,
        'deter_dim': DETER_DIM,
        'hidden_dim': HIDDEN_DIM,
        'buffer_alpha': BUFFER_ALPHA,
        'num_superpose': NUM_SUPERPOSE,
        'interference_strength': INTERFERENCE_STRENGTH,
        'seeds': EXPERIMENT_SEEDS
    },
    'final_performance': {
        'loss_mean': float(np.mean(final_losses)),
        'loss_std': float(np.std(final_losses)),
        'test_mse_mean': float(np.mean(test_mses)),
        'test_mse_std': float(np.std(test_mses))
    },
    'raw_results': {
        'test_mses': [float(x) for x in test_mses]
    }
}

results_dir = Path('../experiments/results/superposition_proper')
results_dir.mkdir(parents=True, exist_ok=True)
with open(results_dir / 'complete_metrics.json', 'w') as f:
    json.dump(complete_metrics, f, indent=2)

print(f"Saved to {results_dir}")